# Exercice 2 — Retrouver les bons passages avec numpy

**Prérequis** : chapitres 2 et 3, et l'exercice 1 (ses conversions sont dans le dossier `conversions/`).

Des employés de Veltis posent des questions. Pour chacune, vous devez retrouver **les passages et les documents** qui contiennent la réponse. C'est la partie « recherche » d'un RAG, sans LLM pour l'instant.

Vous allez :

1. découper tous les documents convertis en chunks ;
2. transformer ces chunks en embeddings avec `bge-m3`, servi par vLLM ;
3. retrouver, avec `numpy`, les chunks et les documents les plus proches de chaque question ;
4. **mesurer** la qualité de votre recherche, puis comprendre ses échecs.

Les cellules qui contiennent `# TODO` sont à compléter. Les cellules **Vérification** testent votre travail.

## 0. Configuration

Adaptez si besoin les chemins (relatifs à ce notebook) et l'adresse du service d'embedding.

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"                         # never download models

from pathlib import Path
import numpy as np
import polars as pl

ROOT = Path("..")                                          # course folder, relative to this notebook
CONVERSIONS_DIR = ROOT / "conversions"
EMBEDDER_DIR = ROOT / "models" / "embedder" / "bge-m3"
EMBEDDINGS_URL = "http://localhost:8001/v1"                # vLLM embedding service
MODEL = "bge-m3"
MAX_TOKENS = 512

## Les questions

Voici les questions des employés, avec pour chacune le ou les documents qui contiennent la réponse. Certaines réponses se trouvent dans plusieurs documents : trouver l'un d'eux suffit.

La dernière question n'a **pas de réponse** dans le corpus : on s'en servira à la fin.

In [ ]:
QUESTIONS = [
    {"id": "Q01", "question": "Je commence lundi chez Veltis : à quelle heure et où dois-je me présenter ?",
     "expected_documents": ["guide_onboarding.pdf"]},
    {"id": "Q02", "question": "Pendant combien de temps vais-je être accompagné par un collègue à mon arrivée ?",
     "expected_documents": ["guide_onboarding.pdf"]},
    {"id": "Q03", "question": "Que se passe-t-il si je ne change pas mon mot de passe provisoire ?",
     "expected_documents": ["guide_onboarding.pdf"]},
    {"id": "Q04", "question": "Dois-je suivre une formation sécurité même si je travaille dans un bureau ?",
     "expected_documents": ["guide_onboarding.pdf"]},
    {"id": "Q05", "question": "Combien de jours par semaine puis-je travailler depuis chez moi ?",
     "expected_documents": ["politique_teletravail_2025.docx", "politique_teletravail_2025.pdf"]},
    {"id": "Q06", "question": "Quelle somme reçoit-on chaque mois quand on travaille à domicile ?",
     "expected_documents": ["politique_teletravail_2025.docx", "politique_teletravail_2025.pdf"]},
    {"id": "Q07", "question": "Combien de jours de vacances ai-je droit chaque année ?",
     "expected_documents": ["intranet_faq_rh.html", "guide_onboarding.pdf"]},
    {"id": "Q08", "question": "Je viens travailler à vélo, est-ce que j'ai droit à une indemnité ?",
     "expected_documents": ["grille_salariale_2026.pdf", "intranet_faq_rh.html", "guide_onboarding.pdf"]},
    {"id": "Q09", "question": "Quel est le salaire de départ d'un technicien ?",
     "expected_documents": ["grille_salariale_2026.pdf"]},
    {"id": "Q10", "question": "Combien de vélos ont été vendus en 2025 ?",
     "expected_documents": ["rapport_annuel_2025.pdf", "plan_strategique_2026-2028.pptx"]},
    {"id": "Q11", "question": "Combien a coûté le robot de soudure acheté en 2025 ?",
     "expected_documents": ["rapport_annuel_2025.pdf"]},
    {"id": "Q12", "question": "Le conseil d'administration a-t-il validé la construction d'une deuxième usine ?",
     "expected_documents": ["pv_conseil_administration_2026-03-12_scan.pdf"]},
    {"id": "Q13", "question": "Quel dividende sera proposé aux actionnaires ?",
     "expected_documents": ["pv_conseil_administration_2026-03-12_scan.pdf"]},
    {"id": "Q14", "question": "Dans quels nouveaux pays Veltis veut-elle vendre ses vélos d'ici 2028 ?",
     "expected_documents": ["plan_strategique_2026-2028.pptx"]},
    {"id": "Q15", "question": "Qui dirige le service qualité ?",
     "expected_documents": ["organigramme_et_processus_production.pdf"]},
    {"id": "Q16", "question": "Combien de temps dure le contrôle qualité d'un vélo ?",
     "expected_documents": ["organigramme_et_processus_production.pdf"]},
    {"id": "Q17", "question": "Quand a lieu la fête du personnel ?",
     "expected_documents": ["newsletter_le_fil_2026-02.pdf"]},
    {"id": "Q18", "question": "Pourrai-je manger à la cafétéria fin mars ?",
     "expected_documents": ["intranet_actualites.html"]},
    {"id": "Q19", "question": "Que dois-je faire si je suis malade ?",
     "expected_documents": ["intranet_faq_rh.html"]},
    {"id": "Q20", "question": "Who founded the company, and in which year?",
     "expected_documents": ["presentation_veltis.pdf"]},
    {"id": "Q21", "question": "Quelle distance peut parcourir le vélo familial avec une seule charge ?",
     "expected_documents": ["presentation_veltis.pdf"]},
    {"id": "Q22", "question": "Quel est le salaire de la directrice générale ?",
     "expected_documents": []},
]
print(len(QUESTIONS), "questions")

## 1. Charger les documents convertis

Écrivez une fonction `load_documents(folder)` qui charge tous les fichiers **JSON** du dossier et renvoie un dictionnaire `{nom du fichier d'origine: DoclingDocument}`.

Le nom du fichier d'origine est le nom du JSON sans son extension `.json` : `guide_onboarding.pdf.json` → `guide_onboarding.pdf`.

*Indice* : `DoclingDocument.load_from_json(path)`, et l'attribut `stem` d'un `Path`.

In [ ]:
from docling_core.types.doc import DoclingDocument

def load_documents(folder):
    """Return {original file name: DoclingDocument} for every JSON file in the folder."""
    # TODO
    ...

documents = load_documents(CONVERSIONS_DIR)

In [ ]:
# Check
print(len(documents), "documents loaded:", sorted(documents))
expected = {doc for q in QUESTIONS for doc in q["expected_documents"]}
missing = expected - set(documents)
if missing:
    print("WARNING: expected documents missing from the conversions:", sorted(missing))
else:
    print("OK: all expected documents are present")

## 2. Découper en chunks

Chargez le tokenizer de `bge-m3` et créez un `HybridChunker` limité à `MAX_TOKENS` tokens.

Découpez ensuite **tous** les documents. Rangez les chunks dans une liste `chunks`, où chaque chunk est un dictionnaire :

| Clé | Contenu |
|---|---|
| `document` | le nom du document d'origine |
| `headings` | les titres des sections, réunis en une chaîne avec `" > "` (chaîne vide s'il n'y en a pas) |
| `text` | le texte **contextualisé** du chunk (titres + texte), celui qu'on transformera en embedding |
| `token_count` | le nombre de tokens de ce texte |

In [ ]:
from transformers import AutoTokenizer
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker

# TODO: the tokenizer, the chunker, then the `chunks` list
chunks = []

In [ ]:
# Check
print(len(chunks), "chunks")
assert chunks and set(chunks[0]) == {"document", "headings", "text", "token_count"}
assert max(c["token_count"] for c in chunks) <= MAX_TOKENS, "A chunk exceeds the token limit"
print("OK: chunks ready")

Affichez, avec `polars`, le nombre de chunks et le nombre moyen et maximal de tokens **par document**.

In [ ]:
# TODO

## 3. Calculer les embeddings

Écrivez une fonction `embed(texts, batch_size=32)` qui :

- envoie les textes au service d'embedding **par lots** de `batch_size` textes (un appel par lot, pas un appel par texte) ;
- renvoie une matrice `numpy` de forme (nombre de textes × dimension), dont chaque ligne est **normalisée** (longueur 1).

*Indice* : `range(0, len(texts), batch_size)` donne le début de chaque lot. `np.linalg.norm(matrix, axis=1, keepdims=True)` donne la longueur de chaque ligne.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=EMBEDDINGS_URL, api_key="no-key")

def embed(texts, batch_size=32):
    """Return a numpy matrix with one normalized row per text."""
    # TODO
    ...

chunk_vectors = embed([c["text"] for c in chunks])

In [ ]:
# Check
print("Shape:", chunk_vectors.shape)
assert chunk_vectors.shape[0] == len(chunks), "There must be one vector per chunk"
assert np.allclose(np.linalg.norm(chunk_vectors, axis=1), 1), "The vectors must be normalized"
print("OK: embeddings ready")

## 4. Retrouver les meilleurs chunks

Calculez les embeddings de toutes les questions, puis la matrice `scores` des similarités entre chaque question et chaque chunk : `scores[i, j]` est la similarité entre la question `i` et le chunk `j`.

*Indice* : les vecteurs étant normalisés, la similarité cosinus est un simple produit scalaire. Et un seul produit de matrices calcule toutes les similarités d'un coup.

In [ ]:
# TODO
question_vectors = ...
scores = ...

In [ ]:
# Check
print("Shape:", scores.shape)
assert scores.shape == (len(QUESTIONS), len(chunks))
print("OK: scores computed")

Écrivez une fonction `top_chunks(question_index, k=3)` qui renvoie les `k` chunks les plus proches de la question numéro `question_index`, sous forme d'une liste de couples `(score, chunk)`, du meilleur au moins bon. Affichez-les pour la première question.

*Indice* : `np.argsort()` trie par ordre **croissant**.

In [ ]:
def top_chunks(question_index, k=3):
    """Return the k chunks closest to the question, as a list of (score, chunk)."""
    # TODO
    ...

## 5. Retrouver les meilleurs documents

Plusieurs chunks d'un même document peuvent être proches de la question. Pour classer les **documents**, on donne à chaque document le score de **son meilleur chunk**.

Écrivez une fonction `top_documents(question_index, k=3)` qui renvoie les `k` meilleurs documents, sous forme d'une liste de couples `(document, score)`, du meilleur au moins bon.

In [ ]:
def top_documents(question_index, k=3):
    """Return the k most relevant documents, as a list of (document, score of their best chunk)."""
    # TODO
    ...

## 6. Mesurer la qualité de la recherche

Pour chaque question qui a des documents attendus, déterminez :

- `top1` : le meilleur document trouvé ;
- `found_top1` : vrai si ce meilleur document fait partie des documents attendus ;
- `found_top3` : vrai si au moins un des 3 meilleurs documents fait partie des documents attendus.

Rangez les résultats dans un tableau `polars` nommé `evaluation`, puis affichez le pourcentage de questions réussies en top 1 et en top 3.

In [ ]:
# TODO
evaluation = ...

Affichez les questions **ratées en top 1**, avec le document trouvé à la place.

In [ ]:
# TODO

**Analysez les échecs.** Pour chaque question ratée, ouvrez le Markdown du document attendu dans `conversions/` et cherchez la réponse. Selon les cas, la cause est-elle :

- une information absente du texte extrait (où se trouvait-elle dans le document d'origine ?) ;
- un document « concurrent » qui parle du même sujet ;
- une formulation de la question trop éloignée du texte ;
- autre chose ?

Notez vos conclusions ci-dessous.

*Vos conclusions :*

## 7. Et quand la réponse n'existe pas ?

La question Q22 (« Quel est le salaire de la directrice générale ? ») n'a aucune réponse dans le corpus. Comparez le score de son meilleur chunk avec les scores des meilleurs chunks des autres questions : minimum, moyenne et maximum.

In [ ]:
# TODO

La recherche renvoie-t-elle quand même des chunks pour cette question ? Lesquels, et pourquoi ? Pourrait-on fixer un **seuil** de score en dessous duquel on répondrait « je ne sais pas » ? Quelles seraient les limites de cette idée ?

*Votre réponse :*

## Bonus

**B.1** Recalculez les embeddings des chunks **sans** les titres (le texte brut du chunk, `chunk.text`, au lieu du texte contextualisé), puis refaites l'évaluation. Les titres aident-ils la recherche ?

**B.2** Refaites tout avec `MAX_TOKENS = 128`. Des chunks plus petits améliorent-ils ou dégradent-ils la recherche ?

**B.3** Calculer les embeddings prend du temps. Sauvegardez `chunk_vectors` sur le disque avec `np.save()`, et la liste `chunks` avec `pl.DataFrame(chunks).write_parquet()`, puis rechargez-les sans rien recalculer.

In [ ]:
# Bonus B.1

In [ ]:
# Bonus B.2

In [ ]:
# Bonus B.3